In [ ]:
import pandas as pd
import tifffile
import os
from os.path import join as opj
from glob import glob
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt

In [ ]:
annot_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2"
previous_finished_group = set(range(281))

In [ ]:
all_finished_groups = [int(os.path.basename(x).removesuffix(".tiff").removeprefix("group")) for x in glob(opj(annot_root, "finished", "*.tiff"))]
to_proc_groups = sorted([fg for fg in all_finished_groups if fg not in previous_finished_group])
to_proc_groups

In [ ]:
keys_not_checking = ["Site/Organ", "Barcode", "Diagnosis", "path", "tile", "comment"]
for tpg in tqdm(to_proc_groups):
    group_meta = pd.read_csv(opj(annot_root, "meta", f"group{tpg}_meta.csv"))
    
    
    annot_file = opj(annot_root, "finished", f"group{tpg}.tiff")
    
    group_im = tifffile.imread(annot_file)
    for key, df_b in group_meta.groupby(["UM Accession", "Block"]):
        block_annot = group_im[list(df_b.index), ...]
        if not os.path.exists(opj(annot_root, "meta", f"{key[0]}.{key[1]}_sections_annot_meta.csv")):
            print(f"DF DNE - {key[0]}.{key[1]}")
            continue
                              
        block_specific_meta = pd.read_csv(opj(annot_root, "meta", f"{key[0]}.{key[1]}_sections_annot_meta.csv"))
        
        if "Unnamed: 0" in block_specific_meta.columns:
            block_specific_meta = block_specific_meta.drop(["Unnamed: 0"], axis=1)
        if not (block_specific_meta.drop(keys_not_checking, axis=1).reset_index(drop=True).equals(
                                      df_b.drop(keys_not_checking, axis=1).reset_index(drop=True))):
            print(f"DF MISMATCH - {key[0]}.{key[1]}")
            
        block_annot = [Image.fromarray(x) for x in block_annot]
        block_annot[0].save(opj(annot_root, "block_annot", f"{key[0]}.{key[1]}.tiff"), save_all=True, append_images=block_annot[1:])

In [ ]:
tpg

In [ ]:
group_meta

In [ ]:
group_im.shape